In [ ]:
"""
STEP 1: Dataset Download & Verification
----------------------------------------


Run: python 1_download_dataset.py
"""

import kagglehub
import os
import shutil

# 1. Download latest version of FER-2013 dataset
path = kagglehub.dataset_download("msambare/fer2013")
print("Dataset downloaded at:", path)


project_dataset_dir = os.path.join(os.getcwd(), "dataset")

if not os.path.exists(project_dataset_dir):
    print(f"Copying dataset to local project folder: {project_dataset_dir}")
    shutil.copytree(path, project_dataset_dir)
else:
    print("dataset/ folder already exists, skipping copy.")

# 3. Verify structure
print("\nFolder structure check:")
for root, dirs, files in os.walk(project_dataset_dir):
    depth = root.replace(project_dataset_dir, "").count(os.sep)
    if depth <= 1:
        print("  " * depth + os.path.basename(root) + "/")

print("\nExpected structure:")
print("dataset/train/<emotion_folders>/*.jpg")
print("dataset/test/<emotion_folders>/*.jpg")
print("\nEmotion folders should be: angry, disgust, fear, happy, neutral, sad, surprise")

Using Colab cache for faster access to the 'fer2013' dataset.
Dataset downloaded at: /kaggle/input/fer2013
Copying dataset to local project folder: /content/dataset

Folder structure check:
dataset/
  test/
  train/

Expected structure:
dataset/train/<emotion_folders>/*.jpg
dataset/test/<emotion_folders>/*.jpg

Emotion folders should be: angry, disgust, fear, happy, neutral, sad, surprise


In [ ]:
!nvidia-smi

Sun Aug 30 07:30:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
STEP 2: CNN Model Build + Training (High Accuracy Setup)

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# ---------------- 1. Config ----------------
train_dir = os.path.join("dataset", "train")
test_dir = os.path.join("dataset", "test")

IMG_HEIGHT, IMG_WIDTH = 48, 48
BATCH_SIZE = 64
NUM_CLASSES = 7
EPOCHS = 50

# ---------------- 2. Data Generators (Augmentation) ----------------
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

print("Class indices (label mapping):", train_generator.class_indices)

# ---------------- 3. Class Weights (Imbalance Handling) ----------------
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes,
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# ---------------- 4. CNN Architecture ----------------
model = Sequential([
    # Block 1
    Conv2D(32, (3, 3), padding="same", activation="relu",
           input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    BatchNormalization(),
    Conv2D(32, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.3),

    # Block 3
    Conv2D(128, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    Conv2D(128, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.4),

    # Fully Connected (ANN part)
    Flatten(),
    Dense(256, activation="relu"),
    BatchNormalization(),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

# ---------------- 5. Callbacks ----------------
checkpoint = ModelCheckpoint(
    "optimal_emotion_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=4, min_lr=1e-6, verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
)

# ---------------- 6. Train ----------------
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=test_generator,
    validation_steps=test_generator.samples // BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=[checkpoint, reduce_lr, early_stop],
)

print("\nTraining complete! Best model saved as optimal_emotion_model.h5")

# ---------------- 7. Final Evaluation ----------------
test_loss, test_acc = model.evaluate(test_generator)
print(f"Final Test Accuracy: {test_acc*100:.2f}%")
print(f"Final Test Loss: {test_loss:.4f}")

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
Class indices (label mapping): {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
Class weights: {0: np.float64(1.0266046844269623), 1: np.float64(9.406618610747051), 2: np.float64(1.0010460615781582), 3: np.float64(0.5684387684387684), 4: np.float64(0.8260394187886635), 5: np.float64(0.8491274770777877), 6: np.float64(1.293372978330405)}


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,470,951 (5.61 MB)

...


Epoch 27: finished saving model to optimal_emotion_model.h5
448/448 ━━━━━━━━━━━━━━━━━━━━ 27s 59ms/step - accuracy: 0.5274 - loss: 1.2140 - val_accuracy: 0.5636 - val_loss: 1.1347 - learning_rate: 2.0000e-04
Epoch 28/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.5312 - loss: 1.1113
Epoch 28: val_accuracy did not improve from 0.56362
448/448 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5312 - loss: 1.1113 - val_accuracy: 0.5626 - val_loss: 1.1368 - learning_rate: 2.0000e-04
Epoch 29/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5286 - loss: 1.2208
Epoch 29: val_accuracy improved from 0.56362 to 0.56948, saving model to optimal_emotion_model.h5



Epoch 29: finished saving model to optimal_emotion_model.h5
448/448 ━━━━━━━━━━━━━━━━━━━━ 27s 61ms/step - accuracy: 0.5362 - loss: 1.1873 - val_accuracy: 0.5695 - val_loss: 1.1270 - learning_rate: 2.0000e-04
Epoch 30/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.5469 - loss: 1.1106
Epoch 30: val_accuracy did not improve from 0.56948
448/448 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5469 - loss: 1.1106 - val_accuracy: 0.5688 - val_loss: 1.1278 - learning_rate: 2.0000e-04
Epoch 31/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.5405 - loss: 1.1894
Epoch 31: val_accuracy improved from 0.56948 to 0.57157, saving model to optimal_emotion_model.h5



Epoch 31: finished saving model to optimal_emotion_model.h5
448/448 ━━━━━━━━━━━━━━━━━━━━ 42s 93ms/step - accuracy: 0.5388 - loss: 1.1927 - val_accuracy: 0.5716 - val_loss: 1.1193 - learning_rate: 2.0000e-04
Epoch 32/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 12s 28ms/step - accuracy: 0.4375 - loss: 1.2561
Epoch 32: val_accuracy did not improve from 0.57157
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4375 - loss: 1.2561 - val_accuracy: 0.5713 - val_loss: 1.1200 - learning_rate: 2.0000e-04
Epoch 33/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5471 - loss: 1.1682
Epoch 33: val_accuracy did not improve from 0.57157
448/448 ━━━━━━━━━━━━━━━━━━━━ 27s 61ms/step - accuracy: 0.5442 - loss: 1.1749 - val_accuracy: 0.5677 - val_loss: 1.1358 - learning_rate: 2.0000e-04
Epoch 34/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.6094 - loss: 1.0162
Epoch 34: val_accuracy did not improve from 0.57157
448/448 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6094 - loss: 1.0162 - 


Epoch 35: finished saving model to optimal_emotion_model.h5
448/448 ━━━━━━━━━━━━━━━━━━━━ 40s 63ms/step - accuracy: 0.5467 - loss: 1.1600 - val_accuracy: 0.5841 - val_loss: 1.0895 - learning_rate: 2.0000e-04
Epoch 36/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - accuracy: 0.4844 - loss: 1.2014
Epoch 36: val_accuracy did not improve from 0.58412
448/448 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.4844 - loss: 1.2014 - val_accuracy: 0.5840 - val_loss: 1.0890 - learning_rate: 2.0000e-04
Epoch 37/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5513 - loss: 1.1592
Epoch 37: val_accuracy improved from 0.58412 to 0.59110, saving model to optimal_emotion_model.h5



Epoch 37: finished saving model to optimal_emotion_model.h5
448/448 ━━━━━━━━━━━━━━━━━━━━ 28s 62ms/step - accuracy: 0.5525 - loss: 1.1560 - val_accuracy: 0.5911 - val_loss: 1.0770 - learning_rate: 2.0000e-04
Epoch 38/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.6250 - loss: 1.1572
Epoch 38: val_accuracy did not improve from 0.59110
448/448 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6250 - loss: 1.1572 - val_accuracy: 0.5904 - val_loss: 1.0773 - learning_rate: 2.0000e-04
Epoch 39/50
448/448 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5510 - loss: 1.1615
Epoch 39: val_accuracy did not improve from 0.59110
448/448 ━━━━━━━━━━━━━━━━━━━━ 28s 62ms/step - accuracy: 0.5491 - loss: 1.1535 - val_accuracy: 0.5903 - val_loss: 1.0776 - learning_rate: 2.0000e-04
Epoch 40/50
  1/448 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.5469 - loss: 1.0625
Epoch 40: val_accuracy did not improve from 0.59110
448/448 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5469 - loss: 1.0625 - v

In [ ]:
from google.colab import files
files.download('optimal_emotion_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>